In [5]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("../data/medium.db")
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
conn

## Overview: tables and row counts

In [3]:
# List all tables
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
for t in tables:
    count = conn.execute(f"SELECT count(*) FROM [{t['name']}]").fetchone()[0]
    print(f"{t['name']:20s} — {count} rows")

articles             — 3 rows
scrape_log           — 0 rows


## Browse articles

In [6]:
df = pd.read_sql_query(
    """\
    SELECT 
        url, title, author, newsletter_date, scraped_at, scrape_status, vector_status 
      FROM articles ORDER BY scraped_at DESC
    """,
    conn
)
df

,url,title,author,newsletter_date,scraped_at,scrape_status,vector_status
0,https://medium.com/@jproco/databricks-ceo-just...,,,None,2026-03-05T09:40:41.424428+00:00,full,indexed
1,https://medium.com/@svante.jacobsen/data-story...,,,None,2026-03-02T09:29:57.639848+00:00,full,pending
2,https://medium.com/@cobusgreyling/claude-code-...,,,None,2026-03-02T08:41:02.524352+00:00,full,indexed


## Status breakdown

In [7]:
print("Scrape status:")
print(df["scrape_status"].value_counts().to_string())
print()
print("Vector status:")
print(df["vector_status"].value_counts().to_string())

Scrape status:
scrape_status
full    3

Vector status:
vector_status
indexed    2
pending    1


## Preview article content

In [ ]:
# Preview the first article's markdown (first 500 chars)
row = conn.execute("SELECT title, raw_markdown FROM articles LIMIT 1").fetchone()
if row:
    print(f"Title: {row['title']}\n")
    print(row["raw_markdown"][:500] if row["raw_markdown"] else "(no content)")

## Scrape log

In [ ]:
pd.read_sql_query("SELECT * FROM scrape_log ORDER BY processed_at DESC", conn)

---
# ChromaDB Vector Store

In [11]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

CHROMA_PATH = Path("../data/chroma")
COLLECTION_NAME = "medium_articles"

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
client

In [13]:
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=DefaultEmbeddingFunction(),
    metadata={"hnsw:space": "cosine"},
)

collection.name, collection.count() # number of chunks in the collection

('medium_articles', 28)

## Browse stored chunks

In [16]:
# Get all chunks with metadata
all_data = collection.get(include=["metadatas", "documents"])

chunks_df = pd.DataFrame({
    "id": all_data["ids"],
    "url": [m.get("url", "") for m in all_data["metadatas"]],
    "title": [m.get("title", "") for m in all_data["metadatas"]],
    "chunk_index": [m.get("chunk_index", 0) for m in all_data["metadatas"]],
    "chunk_length": [len(d) for d in all_data["documents"]],
})

print(f"Chunks per article:")
chunks_df.groupby("url")["id"].count()

Chunks per article:


url
https://medium.com/@cobusgreyling/claude-code-agent-teams-ca3ec5f2d26a                                               14
https://medium.com/@jproco/databricks-ceo-just-dropped-the-most-honest-advice-about-the-future-of-ai-47cda3dd7181    14
Name: id, dtype: int64

In [18]:
print(f"Total chunks: {len(chunks_df)}")
chunks_df.head(10)
# title is missing somehow

Total chunks: 28


,id,url,title,chunk_index,chunk_length
0,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,0,3200
1,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,1,3200
2,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,2,3200
3,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,3,3200
4,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,4,3200
5,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,5,3200
6,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,6,3200
7,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,7,3200
8,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,8,3200
9,https://medium.com/@cobusgreyling/claude-code-...,https://medium.com/@cobusgreyling/claude-code-...,,9,3200


## Semantic search

In [19]:
QUERY = "AI agents"  # <-- change this to search for anything
N_RESULTS = 5

results = collection.query(
    query_texts=[QUERY],
    n_results=min(N_RESULTS, collection.count()),
    include=["documents", "metadatas", "distances"],
)

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"[dist={dist:.4f}] {meta.get('title', '(no title)')} — chunk {meta.get('chunk_index', '?')}")
    print(f"  URL: {meta.get('url', '')}")
    print(f"  {doc[:200]}…\n")

[dist=0.2938]  — chunk 5
  URL: https://medium.com/@cobusgreyling/claude-code-agent-teams-ca3ec5f2d26a
  _Chief Evangelist_**](https://www.linkedin.com/in/cobusgreyling/)**_@_**[_Kore.ai_](https://blog.kore.ai/cobus-greyling/the-shifting-vocabulary-of-ai//?utm_medium=OrganicSocial&utm_source=Medium&utm_c…

[dist=0.3927]  — chunk 1
  URL: https://medium.com/@cobusgreyling/claude-code-agent-teams-ca3ec5f2d26a
  peration=register&redirect=https%3A%2F%2Fcobusgreyling.medium.com%2Fclaude-code-agent-teams-ca3ec5f2d26a&source=---header_actions--ca3ec5f2d26a---------------------post_audio_button------------------)…

[dist=0.4320]  — chunk 8
  URL: https://medium.com/@cobusgreyling/claude-code-agent-teams-ca3ec5f2d26a
  irc--ca3ec5f2d26a----2---------------------2ae23c3d_4a8c_4007_964a_c31d0f7a24ec--------------)

[Five Major Challenges in AI Agents Development ---------------------------------------------- ### Devel…

[dist=0.5563]  — chunk 2
  URL: https://medium.com/@cobusgreyling/claude-cod

In [ ]:
conn.close()